In [ ]:
# ============================================================
#  LLM-Powered RAG System using OpenAI API and ChromaDB
# Description:
#   This script implements a simple Retrieval-Augmented Generation (RAG) pipeline
#   using OpenAI’s LLMs, ChromaDB as a vector database, and PyMuPDF for PDF text extraction.
#   The goal is to extract text from a PDF, create embeddings, store them in ChromaDB,
#   and answer user questions based on document context.
# ============================================================

# --- Install required dependencies (for Colab use) ---
# !pip install -q chromadb openai python-dotenv PyMuPDF
# Setup

!pip install -q chromadb openai python-dotenv PyMuPDF

import os
import sys
from typing import List, Dict
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
import fitz  # PyMuPDF

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 12.1 MB/s eta 0

In [ ]:


#Configure API Key (set manually in Colab)
# Set your OpenAI API key (replace "sk-..." with your actual key in Colab)
os.environ["OPENAI_API_KEY"] = ""  #  Replace with your key

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    print("Error: OPENAI_API_KEY not found")
    sys.exit(1)

#Initialize OpenAI & ChromaDB Clients


try:
   # --- Define embedding function using OpenAI model ---
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=openai_key,
        model_name="text-embedding-3-small"
    )
 # --- Create ChromaDB client (ephemeral in Colab) ---
    chroma_client = chromadb.Client()
     # --- Create or get collection for document QA ---
    collection = chroma_client.get_or_create_collection(
        name="document_qa_collection",
        embedding_function=openai_ef
    )
    # --- Initialize OpenAI client ---
    client = OpenAI(api_key=openai_key)

    print(" Clients initialized successfully")

except Exception as e:
    print(f" Error initializing clients: {e}")
    sys.exit(1)


# ---------------------------------------------------
# --- Extract text content from PDF using PyMuPDF ---
# -------------------------------------------------------

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract full text from a PDF file."""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text("text")
        doc.close()
        print(f" Extracted {len(text)} characters from PDF.")
        return text
    except Exception as e:
        print(f" Error reading PDF: {e}")
        return ""
# -----------------------------------------------------------
# --- Split long text into smaller overlapping chunks ---
# ----------------------------------------------------------------

def split_text(text: str, chunk_size: int = 1000, chunk_overlap: int = 20) -> List[str]:
    """Split text into overlapping chunks."""
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks
# ------------------------------------------------------------
# --- Generate vector embeddings using OpenAI Embedding API ---
# ---------------------------------------------------------------

def get_openai_embedding(text: str) -> List[float]:
    """Generate embeddings using OpenAI API."""
    try:
        response = client.embeddings.create(
            input=text,
            model="text-embedding-3-small"
        )
        return response.data[0].embedding
    except Exception as e:
        print(f" Error generating embedding: {e}")
        return []

#----------------------------------------------------------
# --- Search ChromaDB for the most relevant text chunks ---
# ---------------------------------------------------------
def query_documents(question: str, n_results: int = 2) -> List[str]:
    """Query stored document embeddings."""
    try:
        results = collection.query(query_texts=[question], n_results=n_results)
        return [doc for sublist in results["documents"] for doc in sublist]
    except Exception as e:
        print(f" Error querying documents: {e}")
        return []
# ----------------------------------------------------------
# --- Generate concise and context-aware answer using LLM ---
# ---------------------------------------------------------

def generate_response(question: str, relevant_chunks: List[str]) -> str:
    """Generate concise answer using context."""
    if not relevant_chunks:
        return "No relevant information found."
    try:
        context = "\n\n".join(relevant_chunks)
        prompt = (
            "You are an assistant for question-answering tasks. Use the following pieces of "
            "retrieved context to answer the question. If you don't know the answer, say that you "
            "don't know. Use three sentences maximum and keep the answer concise.\n\n"
            f"Context:\n{context}\n\nQuestion:\n{question}"
        )

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": question},
            ],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" Error generating response: {e}")
        return "Error generating response."

# -------------------------------------------------------
# --- Main pipeline: extract, embed, store, and query PDF ---
# -----------------------------------------------------------

def main():
    """Main PDF ingestion and query pipeline."""
    pdf_path = "Coinbase.pdf"  #  your uploaded file
    text = extract_text_from_pdf(pdf_path)
    if not text:
        print(" No text extracted from PDF.")
        return

    chunks = split_text(text)
    print(f" Created {len(chunks)} text chunks for embedding.")

    for i, chunk in enumerate(chunks):
        embedding = get_openai_embedding(chunk)
        if embedding:
            try:
                collection.upsert(
                    ids=[f"chunk_{i+1}"],
                    documents=[chunk],
                    embeddings=[embedding]
                )
            except Exception as e:
                print(f" Error upserting chunk {i+1}: {e}")

    print(" PDF content indexed successfully.\n")

    # Example query
    question = "How much ransom money did the hackers demand from Coinbase?"
    print(" Query:", question)
    relevant_chunks = query_documents(question)
    if relevant_chunks:
        answer = generate_response(question, relevant_chunks)
        print("\n Answer:", answer)
    else:
        print(" No relevant chunks found.")


# Run the full pipeline
main()


 Clients initialized successfully
 Extracted 2784 characters from PDF.
 Created 3 text chunks for embedding.
 PDF content indexed successfully.

 Query: How much ransom money did the hackers demand from Coinbase?

 Answer: The hackers demanded a $20 million ransom from Coinbase, as reported in the blog post by the company. However, Coinbase stated that they will not pay the ransom and are offering a $20 million reward for information leading to the arrest of the criminals responsible for the cyberattack.
